In [2]:
import json
import os
import glob

# Define paths
locales_dir = "../../i18n/locales"
cards_i18n_path = "../../data/cards_i18n.json"

# Read the existing cards_i18n.json file or initialize an empty array
if os.path.exists(cards_i18n_path):
    with open(cards_i18n_path, "r", encoding="utf-8") as f:
        cards_i18n = json.load(f)
else:
    cards_i18n = []

# Get all locale files
locale_files = glob.glob(f"{locales_dir}/*.json")
print(f"Found {len(locale_files)} locale files: {[os.path.basename(f) for f in locale_files]}")

# Track stats for reporting
total_updated = 0
locale_stats = {}

# Process each locale file
for locale_file in locale_files:
    locale_code = os.path.basename(locale_file).split('.')[0]  # Extract locale code from filename
    
    # Read the locale file
    with open(locale_file, "r", encoding="utf-8") as f:
        locale_data = json.load(f)
    
    # Extract the cards content
    cards_data = locale_data.get("cards", {})
    
    if not cards_data:
        print(f"No card translations found in {locale_code} locale")
        continue
    
    locale_updates = 0
    
    # Update each card with translations for this locale
    for card_id, translation in cards_data.items():
        # Find the card with matching id in the cards_i18n array
        card_found = False
        for card in cards_i18n:
            if card.get("id") == card_id:
                # If card found, update or add the translation for this locale
                if "translations" not in card:
                    card["translations"] = {}
                card["translations"][locale_code] = translation
                card_found = True
                locale_updates += 1
                break
        
        # If the card is not found, add a new entry
        if not card_found:
            new_card = {
                "id": card_id,
                "translations": {
                    locale_code: translation
                }
            }
            cards_i18n.append(new_card)
            locale_updates += 1
    
    total_updated += locale_updates
    locale_stats[locale_code] = locale_updates
    print(f"Updated {locale_updates} card translations for {locale_code} locale")

# Write the updated data back to the file
with open(cards_i18n_path, "w", encoding="utf-8") as f:
    json.dump(cards_i18n, f, ensure_ascii=False, indent=2)

print(f"\nTotal updates: {total_updated} across {len(locale_stats)} locales")
print("Updates per locale:", locale_stats)

Found 6 locale files: ['ja.json', 'en.json', 'tc.json', 'ko.json', 'id.json', 'th.json']
Updated 845 card translations for ja locale
Updated 845 card translations for en locale
Updated 845 card translations for tc locale
Updated 845 card translations for ko locale
Updated 845 card translations for id locale
Updated 845 card translations for th locale

Total updates: 5070 across 6 locales
Updates per locale: {'ja': 845, 'en': 845, 'tc': 845, 'ko': 845, 'id': 845, 'th': 845}


# Synchronize Translations from All Locales

This notebook syncs all card translations from every locale file in the `i18n/locales` directory into the main `cards_i18n.json` file. It ensures that all translated content is properly included in the central translation file.

## How it works:

1. Reads all locale files (en.json, tc.json, ja.json, etc.)
2. For each locale, extracts card translations from the "cards" section
3. Updates the master cards_i18n.json file with translations from each locale
4. Creates new card entries if they don't exist
5. Reports statistics about updates made for each locale

This script helps maintain a comprehensive translation resource that contains all translated content in one place.